In [9]:
!pip install -U -q transformers sentence-transformers chromadb accelerate

In [10]:
import json
import chromadb

from sentence_transformers import SentenceTransformer

#Carregar Notícias

In [11]:
with open("noticias_2.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

print(f"{len(noticias)} notícias carregadas.")

30 notícias carregadas.


#Carregar modelo de embedding

In [12]:
modelo_embedding = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print("Modelo de embeddings carregado.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Modelo de embeddings carregado.


#Preparar documentos

In [13]:
documentos = []
ids = []
metadados = []

for i, noticia in enumerate(noticias):

    documento = (
        f"{noticia['titulo']}\n"
        f"{noticia['resumo']}"
    )

    documentos.append(documento)

    ids.append(f"{noticia['ativo'].lower()}_{i}")

    metadados.append({
        "ativo": noticia["ativo"],
        "data": noticia["data"],
        "fonte": noticia["fonte"]
    })

#Gerar embeddings

In [14]:
embeddings = modelo_embedding.encode(
    documentos,
    normalize_embeddings=True
).tolist()

print(f"{len(embeddings)} embeddings gerados.")

30 embeddings gerados.


#Criar banco vetorial

In [15]:
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(
        name="noticias_carteira"
    )
except Exception:
    pass

colecao_noticias = chroma_client.create_collection(
    name="noticias_carteira",
    metadata={"hnsw:space": "cosine"}
)

#Indexar no chromaDB

In [16]:
colecao_noticias.add(
    embeddings=embeddings,
    documents=documentos,
    ids=ids,
    metadatas=metadados
)

print(f"{len(documentos)} notícias indexadas em 'noticias_carteira'.")

30 notícias indexadas em 'noticias_carteira'.


#Teste de Recuperação

In [17]:
consulta = "dividendos recentes"

vetor_consulta = modelo_embedding.encode(
    [consulta],
    prompt_name="query",
    normalize_embeddings=True
).tolist()


for ativo in ["PETR4", "MXRF11", "CMIG4"]:

    resultado = colecao_noticias.query(
        query_embeddings=vetor_consulta,
        n_results=2,
        where={"ativo": ativo},
        include=["documents", "distances"]
    )

    print(f"\n[{ativo}]")

    for documento, distancia in zip(
        resultado["documents"][0],
        resultado["distances"][0]
    ):
        print(f"[dist={distancia:.3f}] {documento}")


[PETR4]
[dist=0.480] Petrobras aprova R$ 17,4 bi em proventos
Conselho aprovou R$ 17,4 bilhões em dividendos e JCP como antecipação da remuneração de 2026, equivalente a R$ 1,348 por ação, pagos em duas parcelas em novembro e dezembro de 2026.
[dist=0.511] Pagamento da 2ª parcela de dividendos de PETR4
Petrobras pagou a segunda parcela de proventos, no valor de R$ 0,35048636 por ação, referente à antecipação de junho de 2026.

[MXRF11]
[dist=0.468] MXRF11 divulga dividendos de junho
Fundo fixou R$ 0,10 por cota para junho de 2026, repetindo o valor dos dois meses anteriores. Pagamento em 14/07/2026, data-base 30/06/2026, yield mensal de aproximadamente 1,03% sobre cotação de R$ 9,75.
[dist=0.550] MXRF11 mantém dividendo em julho
Dividendo de R$ 0,10 por cota mantido para julho de 2026, pago em 14/08/2026, data-base 31/07/2026. Yield mensal de 1,04% sobre cotação de R$ 9,58.

[CMIG4]
[dist=0.413] Cemig propõe R$ 1,8 bi em dividendos adicionais
Cemig propôs pagamento adicional de R$ 1,8

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Carrega o LLM aberto localmente
nome_llm = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(nome_llm)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    nome_llm,
    dtype=torch.float16,   # a T4 não tem bf16 nativo, então usamos fp16
    device_map="auto",     # coloca o modelo na GPU automaticamente
)

In [23]:
SYSTEM_PROMPT_CARTEIRA = (
    "Você é um assistente de análise de carteiras. Você recebe a posição do "
    "investidor em um ativo e notícias recentes sobre esse ativo. "
    "Sua tarefa é resumir, em até 5 frases, os fatores das notícias que podem "
    "afetar a performance do ativo — tanto pontos positivos quanto riscos. "
    "Use SOMENTE informações que estejam nas notícias fornecidas. "
    "Não invente números, datas ou eventos que não estejam no contexto. "
    "Nunca recomende comprar, vender ou manter o ativo — isso não é "
    "recomendação de investimento, apenas um resumo informativo. "
    "Se as notícias fornecidas não tiverem relação com o que foi perguntado, "
    "diga isso explicitamente."
)

def analisar_posicao(ativo, quantidade, preco_medio, preco_atual=None, n_noticias=4):
    query = f"notícias recentes que podem afetar a performance de {ativo}"
    vetor = modelo_embedding.encode(
        [query], prompt_name="query", normalize_embeddings=True
    ).tolist()
    busca = colecao_noticias.query(
        query_embeddings=vetor,
        n_results=n_noticias,
        where={"ativo": ativo},
        include=["documents", "distances"],
    )
    trechos = busca["documents"][0]
    distancias = busca["distances"][0]

    # Dados da posição
    valor_investido = quantidade * preco_medio
    linhas_posicao = [
        f"Ativo: {ativo}",
        f"Quantidade: {quantidade}",
        f"Preço médio: R$ {preco_medio:.2f}",
        f"Valor investido: R$ {valor_investido:.2f}",
    ]
    if preco_atual is not None:
        variacao_pct = (preco_atual / preco_medio - 1) * 100
        linhas_posicao.append(f"Preço atual: R$ {preco_atual:.2f}")
        linhas_posicao.append(f"Variação: {variacao_pct:+.2f}%")
    texto_posicao = "\n".join(linhas_posicao)

    print(f"--- Posição ---\n{texto_posicao}\n")
    print("--- Notícias recuperadas ---")
    for trecho, dist in zip(trechos, distancias):
        print(f"  [dist={dist:.3f}] {trecho}")
    print()

    # Monta o prompt de geração
    contexto_noticias = "\n".join(trechos)
    mensagens = [
        {"role": "system", "content": SYSTEM_PROMPT_CARTEIRA},
        {
            "role": "user",
            "content": (
                f"Posição do investidor:\n{texto_posicao}\n\n"
                f"Notícias:\n{contexto_noticias}\n\n"
                "Resuma os fatores relevantes dessas notícias para essa posição."
            ),
        },
    ]
    texto_prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer([texto_prompt], return_tensors="pt").to(modelo_llm.device)
    with torch.no_grad():
        saida = modelo_llm.generate(**entradas, max_new_tokens=300, do_sample=False)
    resposta = tokenizer.decode(
        saida[0][entradas.input_ids.shape[1]:], skip_special_tokens=True
    )

    print("--- ANÁLISE ---")
    print(resposta)
    print("\n[Isto é um resumo informativo gerado por IA, não é recomendação de investimento.]")
    return resposta

In [ ]:
analisar_posicao("PETR4", 10, 40, 50)